# 02 · Train and evaluate one or more runs
**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**. About 25-35 min per run.

| Who | Runs |
|---|---|
| First, once | `pilot` (about 15 min) - send the printed inflation check before the real runs |
| Dawar | `seed1`, `seed2`, `seed3` |
| Armughan | `control`, `qonly`, `dilution`, `rank8` |
| Optional, if time | `fullft160m`, `rank64`, `filler10m` |

Each run: builds the training mix (FineWeb-Edu + contaminated copies), fine-tunes Pythia-410M with LoRA, scores every item,
runs the 5 detectors (Perplexity, Zlib, Min-K%, Min-K%++, FSD) and the 13-gram reference check.

**Output:** `results_<run>.zip` per run. Download, unzip into the repo folder, double-click `commit_and_push.bat`.
Notebook 01's files must already be on GitHub.

In [3]:
# ===== CHOOSE THE RUNS FOR THIS SESSION =====
RUNS_TO_DO = ["pilot"]          # e.g. ["seed1", "seed2", "seed3"]  or  ["control", "qonly", "dilution", "rank8"]

In [ ]:
import io
import sys
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from zipfile import ZipFile

REPO_URL = "https://github.com/DawarAhmed27/DAV-Reseach-DataContamination"
REF = "main"
ARCHIVE_URL = f"{REPO_URL}/archive/refs/heads/{REF}.zip"

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
try:
    with urlopen(ARCHIVE_URL, timeout=60) as response:
        archive = response.read()
except URLError as exc:
    raise RuntimeError(f"Could not download {ARCHIVE_URL}: {exc}") from exc

# Extract each download to its own directory to keep earlier runs available.
from tempfile import mkdtemp
download = Path(mkdtemp(prefix="dav_github_", dir=WORK))
with ZipFile(io.BytesIO(archive)) as project:
    root_name = project.namelist()[0].split("/")[0]
    project.extractall(download)
REPO = download / root_name

required = ['dcl/pipeline.py', 'data/items.csv']
missing = [name for name in required if not (REPO / name).is_file()]
if missing:
    raise FileNotFoundError(f"Files missing from GitHub branch {REF}: {', '.join(missing)}")

sys.path.insert(0, str(REPO))
print("Downloaded project from GitHub:", REPO)


In [ ]:
# --- Libraries that Kaggle does not always have ---
!pip uninstall -y -q torchao 2>/dev/null   # Kaggle's old torchao breaks the latest peft; we do not use it
!pip install -q peft sentence-transformers 2>/dev/null | tail -1
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU T4 in Settings > Accelerator")

In [ ]:
assert (REPO / "data" / "items.csv").exists(), "data/items.csv not found - run notebook 01 and push its outputs first"
from dcl import pipeline as P
import zipfile, traceback
for run in RUNS_TO_DO:
    print("=" * 70)
    try:
        P.run_experiment(run, root=REPO)
        z = WORK / f"results_{run}.zip"
        with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
            for p in (REPO / "results").glob(f"{run}*"):
                zf.write(p, p.relative_to(REPO))
        print("Saved", z)
    except Exception:
        traceback.print_exc()
        print(f"Run {run} failed - copy the error above and send it.")